# 3. LightRAG

[LightRAG](https://arxiv.org/abs/2410.05779) reads every chunk with an LLM and extracts **entities** (people, products, organisations) and **relations** between them. The result is a knowledge graph whose nodes and edges carry text descriptions, each embedded for search.

At query time, the LLM first extracts two sets of keywords from the question:

- **low-level** keywords (specific names) are matched against entities, the *local* view;
- **high-level** keywords (themes) are matched against relations, the *global* view.

`hybrid` mode combines both, as in the WildGraphBench evaluation. The matched entities, relations and their source chunks form the context of the answer.

In [1]:
from dotenv import load_dotenv

from src import config

load_dotenv(config.PROJECT_ROOT / ".env")
print(f"Domain: {config.DOMAIN} | run mode: {config.RUN_MODE.value}")

Domain: technology | run mode: subset


In [2]:
from src.data import load_full_corpus_statistics, load_run_inputs
from src.usage_tracking import UsageLedger, get_ledger_path

SYSTEM_NAME = "lightrag_hybrid"
INDEX_NAME = config.INDEX_NAME_BY_SYSTEM[SYSTEM_NAME]

run_inputs = load_run_inputs(config.DOMAIN, config.RUN_MODE)
ledger = UsageLedger(get_ledger_path(run_inputs.run_directory))
working_directory = run_inputs.run_directory / "indexes" / INDEX_NAME
print(f"{len(run_inputs.questions)} questions, {len(run_inputs.documents)} documents")

9 questions, 23 documents


## 3.1 Configure LightRAG

LightRAG takes two functions: one that calls the LLM and one that embeds text. Both come from `src/usage_tracking.py`, so every call is priced in the ledger. Chunking and concurrency match the other systems. Reranking is switched off because no reranking model is part of this comparison.

In [3]:
import logging

from lightrag import LightRAG, QueryParam
from lightrag.utils import setup_logger

from src.usage_tracking import build_lightrag_embedding_function, build_lightrag_llm_function

setup_logger("lightrag", level="WARNING")
logging.getLogger("nano-vectordb").setLevel(logging.WARNING)

working_directory.mkdir(parents=True, exist_ok=True)
index_is_new = not any(working_directory.iterdir())

rag = LightRAG(
    working_dir=str(working_directory),
    llm_model_func=build_lightrag_llm_function(
        ledger, model=config.INDEXING_MODEL, temperature=config.GENERATION_TEMPERATURE
    ),
    llm_model_name=config.INDEXING_MODEL,
    llm_model_max_async=config.MAX_CONCURRENT_LLM_CALLS,
    embedding_func=build_lightrag_embedding_function(
        ledger, model=config.EMBEDDING_MODEL, embedding_dimension=config.EMBEDDING_DIMENSION
    ),
    embedding_func_max_async=config.MAX_CONCURRENT_LLM_CALLS,
    chunk_token_size=config.CHUNK_SIZE_TOKENS,
    chunk_overlap_token_size=config.CHUNK_OVERLAP_TOKENS,
)
await rag.initialize_storages()

## 3.2 Build the graph

`ainsert` chunks the documents, extracts entities and relations from each chunk, merges duplicates across chunks and embeds everything. This is where most of the LightRAG cost is spent.

If the build is interrupted, running the cell again resumes it: LightRAG skips the documents already processed. Deleting `working_directory` forces a full rebuild.

In [4]:
import time

from src.usage_tracking import Phase, load_indexing_report, save_indexing_report, usage_scope

if load_indexing_report(run_inputs.run_directory, INDEX_NAME):
    print("Index found, reusing it.")
else:
    if index_is_new:
        ledger.discard_records(INDEX_NAME, Phase.INDEXING)
    start_time = time.perf_counter()
    with usage_scope(INDEX_NAME, Phase.INDEXING):
        await rag.ainsert(
            run_inputs.documents["text"].tolist(),
            ids=run_inputs.documents["document_id"].tolist(),
            file_paths=run_inputs.documents["file_name"].tolist(),
        )
    save_indexing_report(
        run_inputs.run_directory,
        INDEX_NAME,
        indexing_time_seconds=time.perf_counter() - start_time,
        document_count=len(run_inputs.documents),
        corpus_token_count=int(run_inputs.documents["token_count"].sum()),
    )

indexing_cost_usd = ledger.total_cost_usd(INDEX_NAME, Phase.INDEXING)
print(f"Indexing cost: ${indexing_cost_usd:.4f}")

Indexing cost: $0.2267


On a subset run, the measured cost gives a first estimate of the full build. Cost grows roughly with the number of corpus tokens, since every chunk goes through the same prompts.

In [5]:
from src.usage_tracking import project_full_run_cost_usd

if config.RUN_MODE is config.RunMode.SUBSET:
    projected_cost_usd = project_full_run_cost_usd(
        subset_cost_usd=indexing_cost_usd,
        subset_token_count=int(run_inputs.documents["token_count"].sum()),
        full_token_count=load_full_corpus_statistics(config.DOMAIN)["token_count"],
    )
    print(f"Projected LightRAG indexing cost on the full corpus: ${projected_cost_usd:.2f}")

Projected LightRAG indexing cost on the full corpus: $4.56


## 3.3 Look at the graph

LightRAG saves the graph as a GraphML file. The most connected entities show what the corpus is about.

In [6]:
import networkx as nx
import pandas as pd

knowledge_graph = nx.read_graphml(working_directory / "graph_chunk_entity_relation.graphml")
print(f"{knowledge_graph.number_of_nodes()} entities, {knowledge_graph.number_of_edges()} relations")

top_entities = pd.DataFrame(
    [
        {
            "entity": node,
            "type": attributes.get("entity_type"),
            "degree": knowledge_graph.degree(node),
            "description": attributes.get("description", "")[:150],
        }
        for node, attributes in knowledge_graph.nodes(data=True)
    ]
).sort_values("degree", ascending=False)
top_entities.head(10)

1266 entities, 1277 relations


,entity,type,degree,description
55,Steam,organization,157,Steam is a digital distribution platform devel...
5,Valve,organization,128,Valve Corporation is a prominent video game de...
617,Bloomberg,organization,76,"Bloomberg is a global financial services, soft..."
195,PC Gamer,organization,51,PC Gamer is a prominent publication dedicated ...
243,Counter-Strike: Global Offensive,content,33,Counter-Strike: Global Offensive (CS:GO) is a ...
148,Valve Corporation,organization,31,Valve Corporation is a video game development ...
751,Gabe Newell,person,27,Gabe Newell is the co-founder and managing dir...
974,GeekWire,organization,26,GeekWire is a technology news website that cov...
849,Products,content,22,A section on Bloomberg that provides informati...
309,PCGamesN,organization,21,PCGamesN is a gaming news website that reporte...


In [7]:
most_connected_entity = top_entities["entity"].iloc[0]
print(f"Relations of {most_connected_entity}:")
for _, neighbour, relation in list(knowledge_graph.edges(most_connected_entity, data=True))[:8]:
    print(f"  -> {neighbour}: {relation.get('description', '')[:120]}")

Relations of Steam:
  -> Valve: The relation between Steam and Valve Corporation is that Valve is both the developer and operator of the Steam platform,
  -> Microsoft: Steam collaborates with Microsoft to bring formerly-console-exclusive games to PC.
  -> Trading Cards: Steam includes Trading Cards as part of its gaming ecosystem.<SEP>Steam includes a Trading Card system that allows users
  -> Fake Developers: Fake developers exploit the Steam platform to profit from illegitimate games.
  -> Gama Network: Gama Network provides information about the Steam platform and its features.
  -> Hentai Puzzle: Hentai Puzzle has been removed from the Steam platform.<SEP>Steam has removed the game Hentai Puzzle from its storefront
  -> BOOBS BATTLEGROUND: BOOBS BATTLEGROUND has been removed from the Steam platform.<SEP>Steam has removed the game BOOBS BATTLEGROUND from its 
  -> PUTIN: PUTIN has been removed from the Steam platform.


## 3.4 See what a query retrieves

`aquery_data` runs the retrieval without generating an answer. It shows the keywords, entities, relations and chunks that the answer model would receive.

In [8]:
example_question = run_inputs.questions[run_inputs.questions["question_type"] == "multi_fact"].iloc[0]
print("Q:", example_question["question"])

with usage_scope(SYSTEM_NAME, Phase.QUERY, question_id="inspection"):
    retrieval = await rag.aquery_data(
        example_question["question"],
        QueryParam(
            mode=config.LIGHTRAG_QUERY_MODE,
            chunk_top_k=config.TOP_K_CHUNKS_BY_QUESTION_TYPE["multi_fact"],
            enable_rerank=False,
        ),
    )
retrieved_data = retrieval.get("data", {})
print("Keywords:", retrieval.get("metadata", {}).get("keywords"))
print({kind: len(items) for kind, items in retrieved_data.items()})
pd.DataFrame(retrieved_data.get("entities", []), columns=["entity_name", "entity_type", "description"]).head()

Q: For indie developers unable to cover the cost, which publisher and crowdfunding platform have proposed to pay the Steam Direct fee on their behalf?


Keywords: {'high_level': ['indie developers', 'publisher', 'crowdfunding platform', 'Steam Direct fee'], 'low_level': []}
{'entities': 42, 'relationships': 40, 'chunks': 5, 'references': 3}


,entity_name,entity_type,description
0,Independent Developers,person,Independent developers are individuals or smal...
1,Steam Direct,method,Steam Direct is a platform that allows develop...
2,Steam Greenlight,concept,Steam Greenlight was a system used by Valve th...
3,Fig,organization,Fig is an organization that promises to assist...
4,USgamer,organization,USgamer is a media outlet that reports on gami...


## 3.5 Answer every question

In [9]:
from src.question_runner import answer_all_questions


async def answer_with_lightrag(question: str, question_type: str) -> str:
    """Answer one question with LightRAG hybrid retrieval."""
    return await rag.aquery(
        question,
        QueryParam(
            mode=config.LIGHTRAG_QUERY_MODE,
            response_type=config.RESPONSE_TYPE,
            chunk_top_k=config.TOP_K_CHUNKS_BY_QUESTION_TYPE[question_type],
            enable_rerank=False,
        ),
    )


lightrag_predictions = await answer_all_questions(
    answer_function=answer_with_lightrag,
    questions=run_inputs.questions,
    system_name=SYSTEM_NAME,
    run_directory=run_inputs.run_directory,
    max_concurrent_questions=config.MAX_CONCURRENT_QUESTIONS,
)
print(lightrag_predictions["pred_answer"].iloc[0][:800])

lightrag_hybrid: 0 answered, 9 to go.


lightrag_hybrid:   0%|          | 0/9 [00:00<?, ?it/s]

In May 2022, a federal judge in Seattle ruled that Valve Corporation must face antitrust litigation regarding claims that its "most favored nation" policies for the Steam distribution platform have led to increased video game prices across the industry. The judge, John C. Coughenour, allowed part of the case to proceed, stating that it is plausible that Valve exploits its market dominance to intimidate and retaliate against developers who sell games for less through other retailers or platforms. This decision marked a significant advancement for the plaintiffs, as it followed previous rulings where the judge had initially sided with Valve. The case is part of a broader legal challenge to the standard commission practices of major sales and app distribution platforms, including those of Ama


## 3.6 What it cost

Each question costs a keyword-extraction call, a few embeddings and the answer call.

In [10]:
await rag.finalize_storages()
ledger_summary = ledger.summarize_by_system_and_phase()
ledger_summary[ledger_summary["system_name"].isin([INDEX_NAME, SYSTEM_NAME])]

,system_name,phase,model,call_count,prompt_tokens,completion_tokens,cost_usd
1,lightrag,indexing,gpt-4o-mini,344,1136693,158292,0.220906
2,lightrag,indexing,text-embedding-3-small,327,287402,0,0.005748
3,lightrag_hybrid,query,gpt-4o-mini,18,184634,3571,0.029838
4,lightrag_hybrid,query,text-embedding-3-small,10,487,0,0.000010


## Next

LightRAG retrieves individual entities and relations. Notebook 04 moves one level up: Microsoft GraphRAG groups entities into communities and summarises each one.